# 02 - Construction du Signal Momentum

## Le signal 12-1

Le **momentum 12-1** mesure le rendement sur 12 mois en excluant le dernier mois:

$$\text{Momentum}_t = \frac{P_{t-1}}{P_{t-12}} - 1$$

**Pourquoi exclure le dernier mois ?**  
Le rendement du mois le plus récent est souvent inversé (mean reversion à court terme). L'exclure améliore le signal.

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

from src.data_loader import load_universe_data
from src.signal import compute_momentum_signal, rank_momentum, get_long_short_positions

prices = load_universe_data(verbose=False)

## Calcul du signal

In [ ]:
momentum = compute_momentum_signal(prices, lookback=12, skip=1)

print(f"Période: {momentum.dropna().index[0].date()} à {momentum.index[-1].date()}")
print(f"Momentum moyen: {momentum.mean().mean():.2%}")
momentum.tail()

In [ ]:
# Distribution du signal
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distribution globale
all_mom = momentum.values.flatten()
all_mom = all_mom[~np.isnan(all_mom)]

axes[0].hist(all_mom, bins=50, alpha=0.7, color='steelblue', edgecolor='black')
axes[0].axvline(0, color='red', linestyle='--')
axes[0].set_title('Distribution du momentum 12-1')
axes[0].set_xlabel('Momentum')

# Spread top-bottom
top10_mom = momentum.apply(lambda x: x.nlargest(10).mean(), axis=1)
bot10_mom = momentum.apply(lambda x: x.nsmallest(10).mean(), axis=1)
spread = top10_mom - bot10_mom

spread.dropna().plot(ax=axes[1], color='steelblue')
axes[1].axhline(spread.mean(), color='red', linestyle='--', label=f'Moyenne: {spread.mean():.2%}')
axes[1].set_title('Spread momentum (Top 10 - Bottom 10)')
axes[1].set_ylabel('Spread')
axes[1].legend()

plt.tight_layout()
plt.show()

## Classement cross-sectionnel

Chaque mois, on classe les actions par momentum:
- **Long**: Top 10 (rang le plus élevé)
- **Short**: Bottom 10 (rang le plus bas)

In [ ]:
ranks = rank_momentum(momentum)

# Positions long et short
long_pos, short_pos = get_long_short_positions(momentum, n_long=10, n_short=10)

# Exemple pour le dernier mois
last_date = momentum.dropna().index[-1]
print(f"Positions au {last_date.date()}:")
print(f"\nLONG (Top 10):")
for ticker in long_pos.loc[last_date][long_pos.loc[last_date]].index:
    print(f"  {ticker}: momentum = {momentum.loc[last_date, ticker]:.2%}")

print(f"\nSHORT (Bottom 10):")
for ticker in short_pos.loc[last_date][short_pos.loc[last_date]].index:
    print(f"  {ticker}: momentum = {momentum.loc[last_date, ticker]:.2%}")

In [ ]:
# Stabilité des positions dans le temps
def compute_persistence(positions, window=3):
    """Calcule le % d'actions qui restent dans le portefeuille sur N mois consécutifs."""
    persistence = []
    for i in range(window-1, len(positions)):
        current = set(positions.iloc[i][positions.iloc[i]].index)
        prev = set(positions.iloc[i-window+1][positions.iloc[i-window+1]].index)
        if len(current) > 0:
            overlap = len(current & prev) / len(current)
            persistence.append(overlap)
    return np.mean(persistence)

print(f"Persistance des positions (3 mois):")
print(f"  Long: {compute_persistence(long_pos, 3):.1%}")
print(f"  Short: {compute_persistence(short_pos, 3):.1%}")

## Visualisation de l'évolution des rangs

In [ ]:
# Evolution des rangs pour quelques actions
sample_tickers = ['AAPL', 'MSFT', 'JPM', 'BA', 'WBA']

fig, ax = plt.subplots(figsize=(14, 6))
for ticker in sample_tickers:
    ranks[ticker].dropna().plot(ax=ax, label=ticker, alpha=0.8)

ax.axhline(20, color='green', linestyle='--', alpha=0.5, label='Zone Long')
ax.axhline(10, color='red', linestyle='--', alpha=0.5, label='Zone Short')
ax.fill_between(ranks.index, 20, 30, alpha=0.1, color='green')
ax.fill_between(ranks.index, 0, 10, alpha=0.1, color='red')

ax.set_title('Évolution des rangs momentum')
ax.set_ylabel('Rang (1=pire, 30=meilleur)')
ax.legend(loc='upper right')
plt.tight_layout()
plt.show()

## Validation: absence de look-ahead bias

Le signal au mois t utilise uniquement les prix de t-12 à t-1.  
La position est prise au début du mois t+1.

In [ ]:
# Vérification
test_date = '2023-12-31'
test_ticker = 'AAPL'

mom_value = momentum.loc[test_date, test_ticker]

# Recalcul manuel
p_t_1 = prices.loc[:test_date].iloc[-2][test_ticker]  # Prix à t-1
p_t_12 = prices.loc[:test_date].iloc[-13][test_ticker]  # Prix à t-12
mom_manual = p_t_1 / p_t_12 - 1

print(f"Signal momentum pour {test_ticker} au {test_date}:")
print(f"  Calculé par la fonction: {mom_value:.4f}")
print(f"  Calculé manuellement: {mom_manual:.4f}")
print(f"  Prix utilisés: P(t-1)={p_t_1:.2f}, P(t-12)={p_t_12:.2f}")
print(f"\n=> Aucune donnée future utilisée ✓")

## Points clés

1. Le signal 12-1 capture la tendance à moyen terme
2. Le spread top-bottom varie dans le temps (plus élevé = plus d'opportunités)
3. Les positions sont relativement stables (~50-60% de persistance sur 3 mois)
4. Aucun look-ahead bias: signal calculé avec données passées uniquement